# gerbil-data: Feature Engineering Pipeline for Recommender Systems

End-to-end demo using the MovieLens 1M dataset.

In [1]:
import os, json, subprocess, shutil, pandas as pd

PROJECT_HOME = os.getcwd()
while not os.path.exists(os.path.join(PROJECT_HOME, "pom.xml")):
    PROJECT_HOME = os.path.dirname(PROJECT_HOME)
DATA_DIR = f"{PROJECT_HOME}/examples/ml-1m-small"
OUTPUT_DIR = f"{PROJECT_HOME}/examples/ml-1m-small/train_sample"
JAR = os.path.join(PROJECT_HOME, "target/gerbil-data-1.0.0-jar-with-dependencies.jar")

assert os.path.exists(DATA_DIR), f"Dataset not found at {DATA_DIR}"
assert os.path.exists(JAR), f"JAR not found at {JAR}. Run: mvn clean package -DskipTests"
print(f"Project: {PROJECT_HOME}\nData:    {DATA_DIR}\nJAR:     {JAR}")

Project: /Users/dazhang/PycharmProject/gerbil-data
Data:    /Users/dazhang/PycharmProject/gerbil-data/examples/ml-1m-small
JAR:     /Users/dazhang/PycharmProject/gerbil-data/target/gerbil-data-1.0.0-jar-with-dependencies.jar


## 1. Raw Data

In [2]:
ratings = pd.read_csv(os.path.join(DATA_DIR, "ratings.dat"), sep="::",
    names=["UserID","MovieID","Rating","Timestamp"], engine="python", encoding="latin-1")
print(f"Ratings: {ratings.shape[0]:,} rows, {ratings.UserID.nunique():,} users, {ratings.MovieID.nunique():,} movies")
ratings.head()

Ratings: 500 rows, 6 users, 416 movies


,UserID,MovieID,Rating,Timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291


In [3]:
users = pd.read_csv(os.path.join(DATA_DIR, "users.dat"), sep="::",
    names=["UserID","Gender","Age","Occupation","ZipCode"], engine="python", encoding="latin-1")
print(f"Users: {users.shape[0]:,} rows")
users.head()

Users: 6,040 rows


,UserID,Gender,Age,Occupation,ZipCode
0,1,F,1,10,48067
1,2,M,56,16,70072
2,3,M,25,15,55117
3,4,M,45,7,02460
4,5,M,25,20,55455


In [4]:
movies = pd.read_csv(os.path.join(DATA_DIR, "movies.dat"), sep="::",
    names=["MovieID","Title","Genres"], engine="python", encoding="latin-1")
print(f"Movies: {movies.shape[0]:,} rows")
movies.head()

Movies: 3,883 rows


,MovieID,Title,Genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy


## 2. ETL Pipeline (Spark)

In [5]:
BASE_SPARK_CONF = ["--master", "local[*]", "--driver-memory", "4g",
                   "--conf", "spark.default.parallelism=4",
                   "--conf", "spark.sql.shuffle.partitions=4",
                   "--conf", "spark.driver.extraJavaOptions=-XX:ReservedCodeCacheSize=512m"]

def spark_run(cls, *extra):
    cmd = ["spark-submit"] + BASE_SPARK_CONF + ["--class", cls, JAR, DATA_DIR] + list(extra)
    shell_cmd = "source bash/conf/env.sh && " + subprocess.list2cmdline(cmd)
    r = subprocess.run(shell_cmd, capture_output=True, text=True, cwd=PROJECT_HOME, shell=True)
    ok = r.returncode == 0
    print(f"  {'OK' if ok else 'FAIL'} {cls.split('.')[-1]}")
    if not ok:
        print(f"    {r.stderr[-300:]}")
    return ok

print("Running ETL stages...")
for d in ["clean_sample", "movie_feature", "user_movie_rate_sequence", "join_sample"]:
    p = os.path.join(DATA_DIR, d)
    if os.path.exists(p):
        shutil.rmtree(p)
for cls in ["processing.clean.ML1MCleanSample",
            "processing.feature.ML1MMovieStatFeature",
            "processing.feature.ML1MUserMovieRateSequence",
            "processing.join.ML1MJoinSample"]:
    spark_run(cls)

Running ETL stages...
  OK ML1MCleanSample
  OK ML1MMovieStatFeature
  OK ML1MUserMovieRateSequence
  OK ML1MJoinSample


## 3. Joined Feature Table

In [6]:
join_path = os.path.join(DATA_DIR, "join_sample")
part = sorted(f for f in os.listdir(join_path) if f.endswith('.csv') and not f.startswith('.'))[0]
df = pd.read_csv(os.path.join(join_path, part), sep='\t', nrows=3, escapechar='\\',
    names=["uid","iid","ts","rating","day","user_prof","item_feat","user_beh"])
up = json.loads(df.iloc[0]["user_prof"])
print("User profile:", json.dumps(up, indent=2))
mf = json.loads(df.iloc[0]["item_feat"])
print("\nMovie feature:", json.dumps(mf, indent=2))

User profile: {
  "user_id": "1",
  "gender": "F",
  "age": "1",
  "occupation": "10",
  "zip_code": "48067"
}

Movie feature: {
  "movie_title": "E.T. the Extra-Terrestrial (1982)",
  "movie_genres": "Children's|Drama|Fantasy|Sci-Fi",
  "movie_genre_cnt": "4",
  "movie_rate_count": "2",
  "movie_avg_rate": "4.0",
  "movie_hot_rank": "33"
}


## 4. Featurization (Spark)

In [7]:
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)

cmd = ["spark-submit", "--master", "local[*]", "--driver-memory", "4g",
       "--conf", "spark.serializer=org.apache.spark.serializer.JavaSerializer",
       "--conf", "spark.default.parallelism=4",
       "--conf", "spark.sql.shuffle.partitions=4",
       "--conf", "spark.driver.extraJavaOptions=-XX:ReservedCodeCacheSize=512m",
       "--class", "pipeline.ML1MPipeline",
       JAR, "--yesterday", "20010101", "--parts", "1",
       "--feature_threshold", "1", "--target_threshold", "1",
       "--sample_ratio", "1.0",
       "--input_dir", DATA_DIR, "--output_dir", OUTPUT_DIR,
       "--output_format", "tfrecord"]
shell_cmd = "source bash/conf/env.sh && " + subprocess.list2cmdline(cmd)
print("Running featurization...")
r = subprocess.run(shell_cmd, capture_output=True, text=True, cwd=PROJECT_HOME, shell=True)
for line in r.stdout.split(chr(10)):
    if any(kw in line for kw in ["Total:", "Valid:", "pos_map", "target_map"]):
        print(f"  {line}")
if r.returncode != 0:
    print(f"FAILED: {r.stderr[-300:]}")
else:
    print(f"\nOutput: {OUTPUT_DIR}/20010101/")

Running featurization...
   >>> Transformed | read pos_map size = 0
   >>> Transformed | read target_map size = 0
   >>> Transformed | pos_map_local: 2783. valid feature value count
   >>> Transformed | pos_map: 2841. accumulated feature value count
   >>> Transformed | target_map: 355. accumulated target count
    Total:      400
    Total:      50
    Total:      50
    Total:      400
    Valid:      400 (100.00%)
   >>> Transformed | write pos_map.json path = /Users/dazhang/PycharmProject/gerbil-data/examples/gerbil-output/20010101/pos_map.json
   >>> Transformed | write pos_map.text path = /Users/dazhang/PycharmProject/gerbil-data/examples/gerbil-output/20010101/pos_map.txt
   >>> Transformed | write pos_map.bin path = /Users/dazhang/PycharmProject/gerbil-data/examples/gerbil-output/20010101/pos_map.bin
   >>> Transformed | write pos_map size: 2841
   >>> Transformed | write target_map size: 355

Output: /Users/dazhang/PycharmProject/gerbil-data/examples/gerbil-output/20010101/


## 5. TFRecord Output

In [8]:
from tfrecord.reader import tfrecord_iterator
from tfrecord import example_pb2

tf_dir = os.path.join(OUTPUT_DIR, "20010101", "train", "tfrecord")
files = sorted(f for f in os.listdir(tf_dir) if f.startswith("part-"))
path = os.path.join(tf_dir, files[0])
for i, raw in enumerate(tfrecord_iterator(data_path=str(path), index_path=None)):
    if i >= 1:
        break
    ex = example_pb2.Example()
    ex.ParseFromString(raw)
    print(f"Record {i}: {len(ex.features.feature)} feature fields")
    for name in sorted(ex.features.feature)[:12]:
        feat = ex.features.feature[name]
        kind = feat.WhichOneof('kind').replace('_list', '')
        vals = list(getattr(feat, feat.WhichOneof('kind')).value)[:2]
        print(f"  {name:<30} {kind:<8} {vals}")

Record 0: 175 feature fields
  context_is_weekend_index       int64    [0, 1]
  context_is_weekend_raw         bytes    [b'R:', b'2']
  context_is_weekend_value       float    [1.0, 1.0]
  context_time_area_index        int64    [0, 2]
  context_time_area_raw          bytes    [b'R:', b'3']
  context_time_area_value        float    [1.0, 1.0]
  context_time_hour_index        int64    [0, 5]
  context_time_hour_raw          bytes    [b'R:', b'12']
  context_time_hour_value        float    [1.0, 1.0]
  context_time_week_index        int64    [0, 1]
  context_time_week_raw          bytes    [b'R:', b'7']
  context_time_week_value        float    [1.0, 1.0]


## 6. Vocabulary

In [9]:
with open(os.path.join(OUTPUT_DIR, "20010101", "pos_map.json")) as f:
    vocab = json.load(f)
features = vocab["features"]
print(f"Vocabulary entries: {len(features)}")
for entry in features[:5]:
    print(f"  field={entry['field_name']} index={entry['field_index']} dim={entry['dim']}")

csv_path = os.path.join(OUTPUT_DIR, "20010101", "pos_map.txt")
if os.path.exists(csv_path):
    dims = pd.read_csv(csv_path)
    print(f"\n{dims.shape[0]} feature fields:")
    dims.columns = ["field_name", "index", "type", "dim"]
    print(dims.to_string(index=False, justify="center"))

Vocabulary entries: 58
  field=user_age index=2 dim=5
  field=user_gender index=3 dim=3
  field=user_occupation index=4 dim=6
  field=user_rate_std index=6 dim=4
  field=user_rate_std_7day index=7 dim=4

58 feature fields:
            field_name             index  type  dim
                         user_age    2     1     5 
                      user_gender    3     1     3 
                  user_occupation    4     1     6 
                    user_rate_std    6     1     4 
               user_rate_std_7day    7     1     4 
              user_rate_std_15day    8     1     4 
              user_rate_std_30day    9     1     4 
              user_movie_rate_cnt   10     1     7 
         user_movie_rate_cnt_7day   11     1     7 
        user_movie_rate_cnt_15day   12     1     7 
        user_movie_rate_cnt_30day   13     1     7 
                    user_avg_rate   14     1     4 
               user_avg_rate_7day   15     1     4 
              user_avg_rate_15day   16     1     

## Summary

1. Raw data inspection - complete
2. ETL pipeline - complete (raw data -> joined feature table)
3. Featurization - complete (features -> `{name}_raw/_index/_value` triplets)
4. TFRecord output - complete (ready for TensorFlow training)
5. Vocabulary - complete (embedding position maps for offline/online serving)